In [3]:
# Step 2: Load and Split the Text Documents

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load your document(s)
loader = TextLoader("/home/aditya/dktegenai/d4langchain/sample_docs/hamlet.txt")
documents = loader.load()

# Split into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.split_documents(documents)


In [5]:
# Embed & Create Vector Store

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Local free model from Hugging Face
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store vectors in local Chroma DB
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="rag_chroma_db"
)

vectorstore.persist()


/home/aditya/dktegenai/d4langchain/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_143814/2500095367.py:16: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [10]:
# Set Up Retriever for RAG
retriever = vectorstore.as_retriever()


In [ ]:
# Step 5: Load Mistral via Ollama

from langchain_community.chat_models import ChatOllama

# Use the locally installed mistral model
= ChatOllama(model="mistral")llm 


In [12]:
# Step 6: Combine Retriever & LLM into a RAG Chain

from langchain.chains import RetrievalQA

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)


In [13]:
# Step 7: Ask Questions to Your Knowledge Base

query = "What is this document about?"
result = rag_chain.invoke({"query": query})

print("Answer:\n", result['result'])

print("\nSources:")
for doc in result['source_documents']:
    print(f"📄 {doc.metadata['source']}")


Answer:
  This document appears to be a passage from William Shakespeare's play "Hamlet." It seems to revolve around the sending of a message or greeting (the "dilated Articles") to Norway (Uncle of young Fortinbras), who is apparently bedridden and unaware of his nephew's plans. The messengers are Cornelius and Voltemand, who are instructed to deliver this greeting without any personal authority beyond the scope defined in the documents they carry. The exact nature of the business or the content of the message isn't explicitly stated in this passage. However, it seems that Fortinbras's plans involve a military purpose as suggested by the terms "Leuies," "Lists," and "full proportions" which might refer to levies (armies) or plans for warfare.

Sources:
📄 /home/aditya/dktegenai/d4langchain/sample_docs/hamlet.txt
📄 /home/aditya/dktegenai/d4langchain/sample_docs/hamlet.txt
📄 /home/aditya/dktegenai/d4langchain/sample_docs/hamlet.txt
📄 /home/aditya/dktegenai/d4langchain/sample_docs/hamlet.